In [1]:
!pip install scikit-learn

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.3 MB 1.7 MB/s eta 0:00:05
   --- ------------------------------------ 0.7/8.3 MB 11.1 MB/s eta 0:00:01
   -------- ------------------------------- 1.8/8.3 MB 16.2 MB/s eta 0:00:01
   ------------ --------------------------- 2.5/8.3 MB 16.2 MB/s eta 0:00:01
   ----------------- ---------------------- 3.6/8.3 MB 17.8 MB/s eta 0:00:01
   ---------------------- ----------------- 4.7/8.3 MB 18.8 MB/s eta 0:00:01
   ---------------------------- ----------- 6.0/8.3 MB 20.2 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------  8.3/8.3 MB 22.0 MB/s eta 0:00:01
   ---------------------------------------  8.3/8.3 MB 22.0 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 19.6 MB/s eta 0:00:00
   -----------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import yfinance as yf
import pandas as pd
import numpy as np  # 🌟 新增：导入 numpy 库用来处理数学上的无穷大
from ta.volatility import BollingerBands
from ta.momentum import RSIIndicator
from ta.trend import MACD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import warnings

# 忽略警告信息
warnings.filterwarnings('ignore')

print("="*50)
print("📈 欢迎使用 A股 高阶量化预测引擎 (GBDT版 - 防爆修复版)")
print("提示: 沪市加 .SS (如 603698.SS)，深市加 .SZ (如 000001.SZ)")
print("="*50)
user_ticker = input("👉 请输入股票代码: ")

# ---------------------------------------------------------
# 1. 获取数据与股票名称
# ---------------------------------------------------------
print(f"\n⏳ 正在拉取 {user_ticker} 过去 5 年数据，请稍候...")
df = yf.download(user_ticker, period='5y', progress=False)

ticker_obj = yf.Ticker(user_ticker)
try:
    stock_name = ticker_obj.info.get('shortName', user_ticker)
except:
    stock_name = user_ticker

if df.empty:
    print("❌ 获取数据失败，请检查代码输入是否正确！")
else:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # 提取收盘价和成交量
    df['Close'] = df['Close'].squeeze()
    df['Volume'] = df['Volume'].squeeze()

    # ---------------------------------------------------------
    # 2. 丰富特征工程
    # ---------------------------------------------------------
    print("⚙️ 正在计算 布林带、RSI、MACD 及 动量特征...")
    
    bb = BollingerBands(close=df['Close'], window=20, window_dev=2)
    df['Upper_Band'] = bb.bollinger_hband()
    df['Lower_Band'] = bb.bollinger_lband()
    
    rsi = RSIIndicator(close=df['Close'], window=14)
    df['RSI'] = rsi.rsi()
    
    macd = MACD(close=df['Close'])
    df['MACD_Line'] = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    
    df['Momentum_5D'] = df['Close'].pct_change(periods=5)
    df['Volume_Change'] = df['Volume'].pct_change()

    # ---------------------------------------------------------
    # 3. 🌟 核心修复：处理 A股 停牌导致的无穷大 (Infinity) 🌟
    # ---------------------------------------------------------
    print("🧹 正在清理停牌和极端异常数据...")
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    # 1. 将数据表中所有的 正无穷和负无穷 替换为 NaN (缺失值)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # 2. 现在可以安全地剔除所有包含 NaN 的行了
    ml_df = df.dropna().copy()

    # ---------------------------------------------------------
    # 4. 构建高阶机器学习模型 (GBDT)
    # ---------------------------------------------------------
    print("🧠 正在训练梯度提升树 (Gradient Boosting) 模型...")
    
    X = ml_df[['Close', 'Upper_Band', 'Lower_Band', 'RSI', 
               'MACD_Line', 'MACD_Signal', 'Momentum_5D', 'Volume_Change']]
    y = ml_df['Target']

    split_index = int(len(ml_df) * 0.8)
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

    model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions) * 100

    # ---------------------------------------------------------
    # 5. 预测次日走势
    # ---------------------------------------------------------
    latest_data = X.iloc[-1:]
    tomorrow_predict = model.predict(latest_data)[0]

    latest_close = df['Close'].iloc[-1]
    latest_rsi = df['RSI'].iloc[-1]
    latest_macd = df['MACD_Line'].iloc[-1]

    # ---------------------------------------------------------
    # 6. 格式化输出报告
    # ---------------------------------------------------------
    print("\n" + "★"*50)
    print(f"📊 【{stock_name} ({user_ticker})】 高阶智能预测报告")
    print("★"*50)
    
    print("🔍 [1] 多维特征抓取:")
    print(f"   • 当前盘面 : 收盘价 ￥{latest_close:.2f}")
    print(f"   • 震荡指标 : RSI = {latest_rsi:.2f}")
    print(f"   • 趋势指标 : MACD = {latest_macd:.2f}")
    print(f"   • 其他特征 : 布林带极限、5日动量、成交量异动比")
    
    print("-" * 50)
    print("🎯 [2] 算法模型表现:")
    print(f"   • 驱动内核 : 梯度提升树 (Gradient Boosting)")
    print(f"   • 历史盲测胜率 : {accuracy:.2f}%")
    if accuracy > 53:
        print("   ✅ 评价 : 胜率呈现明显统计学优势 (远超抛硬币)。")
    elif accuracy > 50:
        print("   🟨 评价 : 胜率略具微弱优势，需严格控制仓位。")
    else:
        print("   ⚠️ 评价 : 胜率表现一般，该股票可能近期随机性极强。")
    
    print("-" * 50)
    print("🔮 [3] 次日走势推演:")
    if tomorrow_predict == 1:
        print("   🟢 预测方向 : 【向上变盘 / 看涨】")
        print("   💡 交易参考 : 算法捕捉到潜在的多头共振信号。")
    else:
        print("   🔴 预测方向 : 【向下变盘 / 看跌】")
        print("   💡 交易参考 : 算法提示短期存在抛压或回调风险。")
    print("★"*50 + "\n")

📈 欢迎使用 A股 高阶量化预测引擎 (GBDT版 - 防爆修复版)
提示: 沪市加 .SS (如 603698.SS)，深市加 .SZ (如 000001.SZ)


👉 请输入股票代码:  603698.SS



⏳ 正在拉取 603698.SS 过去 5 年数据，请稍候...
⚙️ 正在计算 布林带、RSI、MACD 及 动量特征...
🧹 正在清理停牌和极端异常数据...
🧠 正在训练梯度提升树 (Gradient Boosting) 模型...

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
📊 【CHANGZHENG ENGINEERING TECH CO  (603698.SS)】 高阶智能预测报告
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔍 [1] 多维特征抓取:
   • 当前盘面 : 收盘价 ￥40.08
   • 震荡指标 : RSI = 56.27
   • 趋势指标 : MACD = 1.50
   • 其他特征 : 布林带极限、5日动量、成交量异动比
--------------------------------------------------
🎯 [2] 算法模型表现:
   • 驱动内核 : 梯度提升树 (Gradient Boosting)
   • 历史盲测胜率 : 52.97%
   🟨 评价 : 胜率略具微弱优势，需严格控制仓位。
--------------------------------------------------
🔮 [3] 次日走势推演:
   🔴 预测方向 : 【向下变盘 / 看跌】
   💡 交易参考 : 算法提示短期存在抛压或回调风险。
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

